# Day 028 · 行情频率与场景
**Frequency Tradeoffs** · 阶段 P1 · 量化基础

> 今天讲行情频率 — 同一只股票,一年下来到底有多少张「照片」可以看?日线一天一张、分钟线一天两百多张、Tick 一天几十万笔、L2 订单簿不仅显示成交还显示每个人的挂单意图。这四种频率本质上是「市场监控录像的帧率」从慢到快的4 档。这一节我们做四件事:① 把四种频率从生活类比角度讲透,让你知道日线 / 分钟线 / Tick / L2 各自适合什么场景;② 看清楚存储 / 算力 / 数据费用 / 策略复杂度怎么随着频率往上升指数级膨胀;③ 学一条永恒铁律 — 高频可以聚合成低频,低频永远还原不出高频;④ 揭开散户最大的迷思 — 「我有 Tick 数据所以我能做高频」为什么 99% 翻车,以及 L2 数据真正的价值不是「看深度」而是「看队列位置」。今天讲完你就能选对自己策略适合的频率,不再迷信「越高频越好」这个伪命题。

---

**课件生成日期:** 2026-05-19  ·  **建议学习时长:** 18 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [ ]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


## 学习目标

- 看懂四种主流行情频率 — 日线 / 分钟线 / Tick / L2 订单簿 — 各自记录什么、适合什么场景
- 理解频率往上走的三重成本 — 存储成本、算力成本、数据订阅费用 — 都是指数级膨胀
- 掌握重采样这条铁律 — 高频聚合成低频用 OHLC + Volume sum,反方向永远做不到
- 看清楚 L2 订单簿真正价值不是「看深度」而是「看队列位置 + 撤单速度」
- 学会按策略类型反推频率需求 — 长线价值用日线、中频技术用分钟、做市套利用 Tick + L2

## 历史背景:从一张照片到一段电影 — 行情频率30 年演化

2000 年之前,中国散户能拿到的行情数据基本只有日线。每天收盘之后,证券公司打印出一张 K 线图寄给客户,或者电视上看上证指数收盘价。一天一个数,一个月30 个数,一年240多个数。整个一年的茅台行情塞进 Excel 一张表轻轻松松。

2008 年前后情况开始变。中国国内开始出现分钟线行情软件,文华财经、通达信、同花顺让散户第一次看到一天内的涨跌情节。一天交易四个小时也就是240 分钟,所以一天240根分钟线。一年240 个交易日乘240就是接近6 万根分钟线,数据量是日线的240 倍。这时候 Excel 已经撑不住了,大家开始用 csv 文件存。

2012 年之后,头部券商开始向少数高净值客户开放 Tick 数据 — 也就是每一笔成交的时间、价格、数量。茅台一天平均成交几万笔,一年就是上千 万笔记录。csv 已经存不下,必须用专门的数据库比如 KDB 或者 InfluxDB。同期境外做市机构早就在毫秒级 Tick 上跑高频策略。

2015 年之后,L2 行情逐步对国内散户开放 — 不只是看成交,而是看10 档买卖盘的实时挂单 / 撤单 / 报价更新。每一档报价每变一次就是一条记录,茅台一天 L2 全量数据轻轻松松上百 万条。这时候个人电脑硬盘存全市场3 个月数据就要几个 T,完整一年要几十个 T。

这30 年从一天一张照片,到一天一段电影,再到一天一部高清纪录片,再到一天一部蓝光院线版,记录信息越多但代价也越大。散户最大的认知偏差是「我能拿到的频率越高,我赚到的钱越多」。这话错得离谱。真相是「频率应该匹配策略,而不是策略追随频率」。长线价值投资者拿日线就够了,跟巴菲特一样几十年盯着年报;高频做市拿 Tick 不够还要 L2 加微秒级时间戳。中间的散户去蹭高频,买几千块 Tick 套餐,回测做出来一条漂亮收益曲线,实盘一上去要么延迟跟不上要么手续费吃光,十有八九亏到怀疑人生。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 四种频率的生活类比 — 监控录像的4 档帧率

想象一下你在自己家门口装一个监控。最便宜的型号一天只拍一张照片 — 每天傍晚拍一张记录今天有没有人来过。中档一点的每分钟拍一帧,你能看到一天里谁来过、谁站了多久。再贵一点的连续录像每秒 30 帧,你能看清来访者的每个动作。最贵的一档不仅录像,还能透过门外人的口袋看清楚他手里拿的是钥匙还是榔头 — 这是「未来意图」的预览。

行情数据的4 档频率完全是这个比喻的金融翻版。日线就是一天一张照片,记录今天的开盘 / 最高 / 最低 / 收盘四个价格加上总成交量,一共五个数。一只股票一年才240多张,文件大小几 KB,像一封短邮件。分钟线是每分钟一帧,记录这一分钟内的开高低收 + 成交量,一天240帧,一年几万根,文件大小几 MB,像一段视频。Tick 是每一笔成交都拍一张,记录成交时刻、价格、数量、买卖方向,茅台一天几万笔,一年几千万条,文件几 GB,像一部高清电影。L2 不仅记录成交,还记录所有10 档买卖盘的挂单变化 / 撤单 / 新增,每条记录都有微秒级时间戳,一只股票一天 L2 全量上百 万条,全市场一天几十 GB,像电影厂的所有底片合集。

这4 档帧率对应四种完全不同的策略生态。长线价值用日线就够,跟巴菲特拿着每年报表慢慢看一样;中频技术分析用分钟线,看日内技术形态、突破回踩、量价配合;高频套利和做市必须用 Tick,捕捉每笔成交背后的方向信号；而要做真正的做市赚价差和队列前列的钱,就必须有 L2 看清楚自己挂的单排在第几位。


### 2. 存储与算力成本 — 频率每上一档,代价指数膨胀

频率往上升一档,数据量不是翻一倍而是几十倍上百 倍。这不是夸张,这是数学事实。

用茅台一只票算一笔账。日线一年240 条 × 每条几十字节,一年几 KB。分钟线一年6 万条 × 每条几十字节,一年几 MB,是日线的两百多倍。Tick 一年上千 万条,一年几 GB,又是分钟线的几百倍。L2 一年上亿条,一年上百 GB,又是 Tick 的几十倍。

这个膨胀不只是硬盘问题。第一,数据订阅费用同步飙升 — 日线绝大多数券商免费送,分钟线少数券商免费送,Tick 一般要订阅几百到几千块一年,L2 全市场年费几万到几十万。第二,算力膨胀 — 跑一个日线策略个人笔记本几秒回测10 年,跑分钟线要几分钟,跑 Tick 回测10 年要几小时,跑 L2 回测一年都要一台几万块的服务器跑一晚。第三,存储与传输 — 个人云盘存 L2 全市场一年要几个 T 空间,光下载就要几天。

业内有一句俗话叫「频率越高门槛越高」。日线门槛极低,人人能玩。分钟线门槛低,几百块软件搞定。Tick 门槛中等,要订阅 + 编程能力 + 像样的服务器。L2 全市场门槛极高,数据费几十万 + 几台高配服务器 + 数据工程师 + 量化研究员 + 接近交易所机房的网络。散户去蹭 L2 是 99% 亏钱,因为投入几万买的数据和几万的电脑跑不过几百万投入的专业机构。


### 3. 重采样铁律 — 高频可以聚合成低频,低频永远还原不出高频

这是一条物理意义上不可逆的铁律,跟时间不能倒流一个道理。

聚合方向 — 高频聚合低频很容易,而且数学上完全严谨。从 Tick 聚合到分钟线,只需要把这一分钟内所有笔的成交价取第一笔作为开盘价,最高价作为最高价,最低价作为最低价,最后一笔作为收盘价,所有成交量相加作为总量,就是标准的 OHLC + Volume sum。从分钟聚合到日线一样的道理,把一天内240根分钟线再做一次 OHLC + Volume sum。这个过程任何编程语言两三行代码就搞定,pandas 一句 resample 就完事。

反方向 — 低频还原成高频根本不可能。日线只告诉你这一天的开高低收四个价,你永远不知道这一天里价格走了多少次反复、什么时候到了最高点、什么时候到了最低点、中间有没有过特殊形态。两支日线长得一模一样的票,分钟线轨迹可以完全不同 — 一只可能开盘冲高后震荡下来,一只可能盘整一天最后5 分钟拉涨停。所以你买了「日线策略」基于日线数据回测,根本回测不了任何依赖盘中行为的逻辑。

实战含义有两条铁律级别的推论。第一,买数据要买你够得着的最高频。买了 Tick 一定能聚合出分钟线和日线,买了日线永远做不出分钟线或 Tick。第二,跨频率回测必须用同频率数据。如果你的策略实盘用分钟线开仓,回测用日线收盘价代替,实盘和回测的成交价必然有几个百分点的偏差,回测越漂亮实盘越翻车。这是新手最容易踩的「频率不一致」陷阱。


### 4. L2 订单簿真价值 — 不是看深度,是看队列位置

L2 是10 档买卖盘的实时挂单 / 撤单 / 新增数据。绝大多数散户接触 L2 之后只盯着「深度」看 — 买盘各档加起来多少股,卖盘各档加起来多少股。这个用法只用到了 L2 价值的两成,而且做法上还经常被虚晃单 spoofing 反向操纵。

L2 真正的价值在两件事上。第一件事是看队列位置 — 也就是我的限价单挂出去之后,在这一档队列里排第几位。前一节我们讲过价格优先 + 时间优先两条撮合铁律,在同一价格档内先挂的先成交。如果我挂的单排在前5 位,下一笔市价单来我大概率成交;如果我排在500 位以后,前面四百九十多笔可能都成交不了我已经被价格甩出去了。能不能算清楚自己排第几位,是做市策略赚不赚到 spread 的核心。

第二件事是看撤单速度 — 也就是某一档挂单平均挂多久就被撤掉。挂上去几秒就撤的多半是虚晃单,挂上去几分钟还在的多半是真实意图。专业做市商每秒几千次更新报价,看 L2 实时数据流就能识别哪些是同行哪些是机构哪些是真散户。识别出来之后选择性应对,这就是 L2 真正给做市行业的 edge。

散户的实战含义有两条。第一,如果你做的是日内中频策略,看到 L2 大单出现别立刻跟风,先看这个大单挂了多久。挂上去一秒不到就撤,八成是虚晃。第二,如果你立志要做做市,光买 L2 数据不够,还要写一个队列位置追踪器 — 在每个时刻知道自己挂的每一单排在所有挂单里第几位。这个追踪器的工程难度比数据本身大10 倍,这才是做市真正的护城河。


### 5. 策略匹配频率 — 按策略反推数据需求

频率不是越高越好,而是要匹配你的策略类型。这是这一节最重要的认知。

第一类策略 — 长线价值投资。持仓周期半年到几年,关心的是公司基本面变化,日线足够甚至每周看一次都行。这种策略买 Tick 数据完全浪费钱,跟去 4S 店保养自行车一样。代表是巴菲特、芒格、张磊 — 他们的电脑里大概率装的是公司财报和年报,不是 L2 行情。

第二类策略 — 中频技术 / 量价分析。持仓周期几小时到几周,关心的是日内或多日的技术形态、突破回踩、量价配合。分钟线最合适,5 分钟到15 分钟级别足以捕捉绝大多数信号。日线信号太少容易过拟合,Tick 数据噪音太大反而干扰判断。代表是大量国内技术派和量化新手 — 他们的电脑里跑的是分钟级回测和实时监控。

第三类策略 — 中频套利 / 配对交易。同时持有多个相关标的对冲,捕捉短期价差异常。分钟线到秒级 Tick 都可以,看具体策略 — 期现套利可以分钟,跨市场套利需要 Tick。这类需要解决数据同步问题,不同市场的时钟必须对齐。

第四类策略 — 高频做市 / 趋势跟随。持仓周期秒到分钟,关心的是订单簿动态和短期价格信号。必须用 Tick + L2,而且必须接近交易所机房放服务器。年成本几十万到几百万,产能要求几亿到几十亿规模才划算。代表是 Citadel、Jane Street、Two Sigma 这种专业机构 — 他们的成本只有规模化之后才能摊薄。

散户最大的错误是「我看到别人用 Tick 赚到钱,我也去买」。前提是别人有几亿规模能摊掉几百万的固定成本,你只有几十万怎么摊。最适合个人散户的是第一类和第二类,日线 + 分钟线已经足够你用一辈子。


## 实操:重采样实验 — 把分钟线聚合成5 分钟 / 15 分钟 / 一小时 / 日线 + 模拟 Tick 演示

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [ ]:
# day_028_frequency_resampling.py — 行情频率与场景对比
# 用腾讯港股 0700.HK 一周分钟线 → 重采样到 5min / 15min / 60min / 1D
# + 模拟 Tick → 重采样回 1min,验证「高频聚合低频」铁律
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
np.random.seed(42)

# ============ 1. 拉取腾讯港股一周分钟线 ============
ticker = '0700.HK'
df_1m = yf.download(ticker, period='5d', interval='1m', progress=False, auto_adjust=False)
if isinstance(df_1m.columns, pd.MultiIndex):
    df_1m.columns = df_1m.columns.get_level_values(0)
df_1m = df_1m[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
print(f'分钟线数据范围 {df_1m.index[0]} → {df_1m.index[-1]}')
print(f'分钟线总条数 {len(df_1m)}')
print(f'存储大小 {df_1m.memory_usage(deep=True).sum() / 1024:.1f} KB')

# ============ 2. 重采样到 5min / 15min / 60min / 1D ============
def resample_ohlcv(df, rule):
    agg = df.resample(rule).agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum'
    }).dropna()
    return agg

df_5m = resample_ohlcv(df_1m, '5min')
df_15m = resample_ohlcv(df_1m, '15min')
df_60m = resample_ohlcv(df_1m, '60min')
df_1d = resample_ohlcv(df_1m, '1D')

print('\n========== 频率重采样结果 ==========')
summary = pd.DataFrame({
    '频率': ['1min', '5min', '15min', '60min', '1D'],
    '条数': [len(df_1m), len(df_5m), len(df_15m), len(df_60m), len(df_1d)],
    '存储 KB': [df.memory_usage(deep=True).sum() / 1024 for df in [df_1m, df_5m, df_15m, df_60m, df_1d]]
})
summary['相对1d倍数'] = summary['条数'] / summary['条数'].iloc[-1]
print(summary.to_string(index=False))

# ============ 3. 形态保留度 — 各级别 close 的相关性 ============
# 把不同频率的 close 重采样回到统一时间网格做对比
close_1m = df_1m['Close']
close_5m_aligned = df_5m['Close'].reindex(close_1m.index, method='ffill')
close_15m_aligned = df_15m['Close'].reindex(close_1m.index, method='ffill')
close_60m_aligned = df_60m['Close'].reindex(close_1m.index, method='ffill')

corr_5m = close_1m.corr(close_5m_aligned)
corr_15m = close_1m.corr(close_15m_aligned)
corr_60m = close_1m.corr(close_60m_aligned)
print(f'\n========== 各级别 close 与 1min 的相关性 ==========')
print(f'5min : {corr_5m:.4f}')
print(f'15min: {corr_15m:.4f}')
print(f'60min: {corr_60m:.4f}')

# ============ 4. 模拟 Tick 数据 — 1 分钟内随机生成 50 笔成交 ============
# 用第一根分钟线的开收价范围作为基准,模拟 Tick 流
sample_minute = df_1m.iloc[0]
tick_open = float(sample_minute['Open'])
tick_close = float(sample_minute['Close'])
tick_high = float(sample_minute['High'])
tick_low = float(sample_minute['Low'])

n_ticks = 50
tick_times = pd.date_range(df_1m.index[0], periods=n_ticks, freq=f'{60//n_ticks*1000}ms')
# 用 GBM 风格游走生成 Tick 价格,确保覆盖 high / low 范围
tick_prices = np.linspace(tick_open, tick_close, n_ticks)
tick_prices += np.random.normal(0, (tick_high - tick_low) * 0.15, n_ticks)
tick_prices = np.clip(tick_prices, tick_low, tick_high)
tick_volumes = np.random.randint(100, 1000, n_ticks)
tick_side = np.random.choice(['B', 'S'], n_ticks, p=[0.5, 0.5])

tick_df = pd.DataFrame({
    'price': tick_prices,
    'volume': tick_volumes,
    'side': tick_side
}, index=tick_times)
tick_df.index.name = 'ts'
print(f'\n========== 模拟 Tick 数据 ==========')
print(f'Tick 条数 {len(tick_df)} · 时间跨度 60 秒')
print(tick_df.head().to_string())

# ============ 5. 把 Tick 聚合回 1 分钟,验证铁律 ============
tick_1m = pd.DataFrame({
    'Open': [tick_df['price'].iloc[0]],
    'High': [tick_df['price'].max()],
    'Low': [tick_df['price'].min()],
    'Close': [tick_df['price'].iloc[-1]],
    'Volume': [tick_df['volume'].sum()]
}, index=[tick_df.index[0]])
print(f'\n========== Tick → 1min 聚合结果 ==========')
print(tick_1m.to_string())
print(f'原 1min Open {tick_open:.2f} High {tick_high:.2f} Low {tick_low:.2f} Close {tick_close:.2f}')

# ============ 6. 画图 — 四级别 K 线对比 + 条数对比柱状图 + Tick 散点 ============
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# (1) 四级别 close 走势对比
axes[0, 0].plot(df_1m.index, df_1m['Close'], color='#1e90ff', linewidth=0.6, label='1min', alpha=0.6)
axes[0, 0].plot(df_5m.index, df_5m['Close'], color='#ff8c00', linewidth=1.0, label='5min')
axes[0, 0].plot(df_15m.index, df_15m['Close'], color='#2e8b57', linewidth=1.3, label='15min')
axes[0, 0].plot(df_60m.index, df_60m['Close'], color='#8b008b', linewidth=1.6, label='60min')
axes[0, 0].set_title('Close (1min / 5min / 15min / 60min)', fontsize=11)
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Price')
axes[0, 0].legend(loc='best', fontsize=9)
axes[0, 0].grid(alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=20)

# (2) 条数对比柱状图
freqs = ['1min', '5min', '15min', '60min', '1D']
counts = [len(df_1m), len(df_5m), len(df_15m), len(df_60m), len(df_1d)]
colors = ['#1e90ff', '#ff8c00', '#2e8b57', '#8b008b', '#dc143c']
bars = axes[0, 1].bar(freqs, counts, color=colors)
axes[0, 1].set_yscale('log')
axes[0, 1].set_title('Data Points per Frequency (log scale)', fontsize=11)
axes[0, 1].set_ylabel('Number of Bars (log)')
for bar, count in zip(bars, counts):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{count}', ha='center', va='bottom', fontsize=9)
axes[0, 1].grid(alpha=0.3, axis='y')

# (3) 模拟 Tick 散点
tick_buy = tick_df[tick_df['side'] == 'B']
tick_sell = tick_df[tick_df['side'] == 'S']
axes[1, 0].scatter(tick_buy.index, tick_buy['price'], s=tick_buy['volume']/10,
                    color='#dc143c', alpha=0.6, label='Buy')
axes[1, 0].scatter(tick_sell.index, tick_sell['price'], s=tick_sell['volume']/10,
                    color='#2e8b57', alpha=0.6, label='Sell')
axes[1, 0].axhline(tick_open, color='gray', linestyle='--', linewidth=0.7, label='Open')
axes[1, 0].axhline(tick_close, color='black', linestyle='--', linewidth=0.7, label='Close')
axes[1, 0].set_title(f'Simulated Tick (50 trades in 1 minute)', fontsize=11)
axes[1, 0].set_xlabel('Tick Time')
axes[1, 0].set_ylabel('Trade Price')
axes[1, 0].legend(loc='best', fontsize=9)
axes[1, 0].grid(alpha=0.3)
axes[1, 0].tick_params(axis='x', rotation=20)

# (4) 各级别 close 与 1min 相关性
corr_data = [1.0, corr_5m, corr_15m, corr_60m]
corr_labels = ['1min', '5min', '15min', '60min']
axes[1, 1].bar(corr_labels, corr_data, color=['#1e90ff', '#ff8c00', '#2e8b57', '#8b008b'])
axes[1, 1].set_ylim(0, 1.05)
axes[1, 1].set_title('Correlation with 1min Close', fontsize=11)
axes[1, 1].set_ylabel('Pearson Correlation')
for i, v in enumerate(corr_data):
    axes[1, 1].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=9)
axes[1, 1].grid(alpha=0.3, axis='y')

plt.suptitle('Frequency Resampling — Tencent 0700.HK 1week',
             fontsize=12, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('chart_01.png', dpi=120, bbox_inches='tight')
print('\n图已保存 chart_01.png')

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| A 股散户买 Tick 套餐 | 某主流数据商 | 某 A 股散户2023 年看了一些「量化教学博主」的鼓吹,花了七千多块订阅了一年的 A 股 Tick 数据,买了一台一万多的高配电脑,自学了 Python 和 pandas,信心满满想做日内做市策略。回测在历史 Tick 数据上做得很漂亮,年化30%收益,夏普比率1.8。实盘半年下来净亏12%。事后复盘发现三个问题:第一,他用的 Tick 数据有两秒延迟,真实做市机构是微秒级,信号到他这里已经过时；第二,他的下单链路也有几百毫秒,等他识别信号 + 下单 + 系统响应,合计接近一秒,所有有 edge 的机会都被专业机构吃光；第三,他的策略频繁交易,半年下来手续费就吃掉了七个百分点的收益。结论是 Tick 数据本身没有错,错的是散户没有跟上的下单链路和规模效应。 |
| 中信证券 L2 行情订阅 | 面向高净值客户 | 中信证券对资产超过50 万的客户开放 L2 10 档买卖盘行情订阅,年费几千块。多数客户买回去之后只把它当成「5 档行情的升级版」用 — 看深度判断支撑阻力。真正会用 L2 的客户极少,需要写软件实时追踪自己的挂单队列位置 + 监测大单撤单速度。中信内部统计显示,订阅 L2 的客户里只有不到5%真正用上了队列位置追踪,其余95%本质上是在花钱买「更多档位」的视觉满足感,实际效益跟普通5 档行情差不多。 |
| Citadel Securities 美股做市 | 全美第一做市商 | Citadel Securities 占美股零售订单流的35%,做的就是「散户下单到券商,券商把订单卖给 Citadel,Citadel 撮合赚 spread + 数据回流」的生意。它能赚到这笔钱靠的就是几件事:第一,买全市场所有交易所的 L2 数据,合并成 NBBO 全市场最佳买卖价;第二,在每个交易所机房放服务器,用 FPGA 硬件做亚微秒级撮合;第三,实时追踪自己挂的每一笔限价单的队列位置,确保大部分挂在 rank 等于一的位置。这套基础设施一年成本几亿美元,Citadel 一年做市收入几十亿。这个生意的本质是「规模 × 速度 × 数据」的乘法,散户哪一个维度都比不了。 |
| Tushare 个人量化平台 | 国内分钟数据替代 | Tushare 是国内散户最常用的量化数据平台之一。Pro 版套餐分几档,日线大部分免费,分钟线几百块一年,Tick 几千块一年。最经典的散户使用场景是花几百块买分钟线 + 几小时写一个回测框架 + 半年跑出一个看起来不错的策略 + 几万本金实盘验证。这条路虽然不能让你发财,但是个非常合理的量化入门路径,投入产出比可控。Tushare 团队自己内部统计过,买 Tick 套餐的散户客户里超过80%在半年内退订回到分钟线,主要原因就是「Tick 数据太多处理不过来,我的策略实际还是分钟级的逻辑」。 |
| Bitcoin 加密做市 | 币安 BTC USDT 永续 | 加密市场24 小时连续交易,Tick 和 L2 数据完全免费,任何人都可以通过 WebSocket 接到币安交易所拿到实时全量数据。这听起来对散户友好,实际上反而是更残酷的内卷。因为数据免费,全世界几千个散户和机构都在做同样的高频套利和做市,导致每个机会的窗口期都是几毫秒甚至几百微秒。一个普通散户从香港或北京通过家用网线连币安,延迟100毫秒以上,而专业机构在新加坡 AWS 机房直接对接币安撮合引擎延迟低于一毫秒。同样的免费数据,延迟差100倍,做市策略胜负完全是基础设施的较量。这就是「数据免费门槛反而高」的悖论。 |


## 常见坑

### ⚠ 01. 迷信「我有 Tick 数据所以我能做高频」

高频策略不只是数据,更是延迟 + 规模 + 算法 + 风控的乘法。散户拿到 Tick 数据只是高频拼图的10%,其余九成压根凑不齐。**正确做法**:散户的最优起点是分钟线 + 中频策略,投入产出比远高于 Tick 策略。先用分钟线练一年,赚到钱再考虑往上走。

### ⚠ 02. 用日线回测分钟级策略

策略实盘用分钟线开仓,回测时只有日线收盘价代替,这是新手最容易犯的「频率不一致」错误。回测看起来很漂亮,实盘成交价完全对不上,误差几个百分点足以让任何策略变成赌博。**正确做法**:回测频率必须严格等于实盘频率。买不起 Tick 就别做 Tick 策略;只有分钟线就老实做分钟级策略。

### ⚠ 03. L2 数据只看深度,不看队列位置

大多数散户买了 L2 之后只看10 档累积深度,跟普通5 档行情用法没区别。L2 真正价值在队列位置追踪和撤单速度检测,这两件事都需要专门写软件实现,数据本身只是原料。**正确做法**:如果你不会写队列位置追踪器,L2 数据对你来说就是过载信息,不如老老实实看5 档。

### ⚠ 04. Tick 数据延迟两秒还以为是实时

市面上大多数面向散户的 Tick 数据源,实际延迟在500 毫秒到两秒之间,专业做市机构是微秒级。这个延迟差对 Tick 策略是致命的,等你看到信号机构早就吃完跑了。**正确做法**:买 Tick 数据之前先做延迟测试,用同一时间戳的官方公告对比自己拿到的 Tick 时间戳,知道你的真实延迟是多少。如果超过100毫秒,基本告别真正的高频策略。

### ⚠ 05. 存全市场 Tick 数据想做大规模回测

全市场 Tick 一年几个 T,光下载就要几天,存储需要专门的时序数据库,日常查询和分析对算力要求极高。散户自己折腾这个,大概率折腾几个月没产出。**正确做法**:做大规模研究的时候用云平台按需付费,跑完结论就清掉;不要自己买几台服务器在家组小机房。除非你已经有稳定盈利的策略需要扩展,普通学习阶段没必要折腾基础设施。

## 实战 SOP · 行情频率与场景选择实战 SOP

1. 策略先于数据 — 先想清楚自己的策略持仓周期,再反推需要多高频的数据,而不是反过来
2. 买数据按你够得着的最高频买 — 反正聚合到低频是几行代码的事,反方向永远做不到
3. 回测频率必须严格等于实盘频率 — 跨频率混用是新手最大的回测过拟合源头
4. L2 看队列位置不看深度 — 不写队列追踪器的话,L2 对散户跟5 档行情没区别
5. Tick 数据先测延迟再下单 — 延迟超过100毫秒就别做真正意义上的高频策略
6. 存储用够就好不要囤 — 个人学习阶段没必要存全市场全频率数据,用云平台按需付费
7. 频率越高手续费杀伤越大 — 高频策略胜负主要看交易成本控制,不是看信号准不准

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 四种行情频率从慢到快是日线 / 分钟线 / Tick / L2 订单簿,本质是市场监控录像的4 档帧率
3. 频率每上一档,数据量从几百倍到几千倍指数膨胀,存储成本 + 算力成本 + 数据费用 + 策略复杂度同步飙升
4. 重采样铁律 — 高频聚合低频用 OHLC + Volume sum,几行 pandas 代码搞定;低频永远还原不出高频
5. L2 订单簿真价值不是看深度,而是看队列位置 + 撤单速度,这两件事都需要专门写软件实现
6. 策略类型决定频率需求 — 长线价值用日线、中频技术用分钟、中频套利用秒级 Tick、高频做市必须 Tick + L2
7. 散户的最优起点是分钟线 + 中频策略 — 投入产出比远高于 Tick / L2,先在分钟线赚到钱再考虑往上走
8. 高频策略的胜负不是数据,而是延迟 + 规模 + 算法 + 风控的乘法,散户哪个维度都比不过专业机构

## 自测题

**Q1.** 为什么高频数据可以聚合成低频,反方向却不行?(提示:OHLC + Volume sum 几行代码;低频丢失了盘中所有轨迹信息)

**Q2.** L2 订单簿和5 档行情最大的差别是什么?L2 真正价值在哪两件事上?(提示:10 档 vs 5 档只是表象,队列位置追踪 + 撤单速度检测才是核心)

**Q3.** 假设你买了一只 A 股的 Tick 数据但发现延迟有两秒,这种数据还能做高频策略吗?为什么?(提示:专业机构微秒级,两秒延迟意味着所有有 edge 的机会都被吃光)

**Q4.** 为什么 Citadel Securities 能从美股零售订单流做市赚到一年几十亿美元?它的护城河是什么?(提示:规模 × 速度 × 数据的乘法,几亿美元年投入的基础设施)

**Q5.** 如果你只有几十万本金想做量化,应该选什么频率作为起点?为什么不是 Tick?(提示:分钟线最划算,Tick 的固定成本规模要几亿才能摊薄)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 029 · 第一阶段实战:画一份策略报告** (Phase 1 Lab)

第29 课讲数据清洗与异常值处理 — 我们已经知道各种频率数据长什么样,接下来用一周时间把脏数据问题搞透。从缺失值 / 异常值 / 重复值 / 时间戳错乱 / 复权前后不一致这五大常见数据病开始,看清楚为什么「80%的量化时间花在数据清洗上」这句话是真的。

## 推荐阅读

- Hasbrouck《Empirical Market Microstructure》(2007,Oxford)— 高频数据分析的开山教科书,讲透不同频率数据的统计性质差异
- O'Hara《Market Microstructure Theory》(1995,Blackwell)— 撮合机制和订单簿动态的理论基础,理解 L2 数据的经济意义
- Cartea, Jaime, Penalva《Algorithmic and High-Frequency Trading》(2015,Cambridge)— 现代高频策略实操指南,讲清楚不同频率适配不同策略类型
- Easley, Lopez de Prado, O'Hara《The Volume Clock — Insights into the High Frequency Paradigm》(Journal of Portfolio Management 2012)— 反思「时间频率」vs「成交量频率」的经典论文,改变了高频策略的取样思维
- Python 工具栈:pandas resample 是聚合频率的标准工具,yfinance 提供日线 + 分钟线免费数据,akshare 提供国内 A 股全频率数据,Tushare Pro 提供专业 Tick 数据(需订阅)